In [1]:
import os, re, gc
import inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

e:\Spbu_ML\spbu_dl_2025\venv_gpt2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
set_seed(42)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "samedad/mem-and-russian-jokes-dataset"
DATASET_SPLIT = "train"

OUTPUT_DIR = "./joke-sft-qwen"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 256
TRAIN_SAMPLES = 30_000
EVAL_SAMPLES  = 1_500

USE_4BIT = False  

MAX_STEPS = 1500
LR = 5e-5

PER_DEVICE_BS = 2
GRAD_ACCUM = 16
PACKING = True

EXPORT_PATH = "joke.txt"
N_JOKES_TO_EXPORT = 600
MAX_NEW_TOKENS = 96


In [ ]:
SYSTEM_PROMPT = "Ты — профессиональный автор анекдотов. Пиши коротко, связно и смешно."
USER_PROMPT = "Напиши ОДИН анекдот на русском в 1–2 предложениях: сначала сетап, потом панчлайн. Только анекдот, без объяснений. Одна строка."


NSFW_BAD = [
    "жоп", "еб", "сук"
]

def is_nsfw(s: str) -> bool:
    low = s.lower()
    return any(w in low for w in NSFW_BAD)


def normalize_one_line(s: str) -> str:
    s = str(s).replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_gpt_joke(example) -> str:
    conv = example.get("conversations", [])
    for msg in conv:
        if isinstance(msg, dict) and msg.get("from") == "gpt":
            return clean_joke_text(msg.get("value", ""))
    return ""


def is_ok_joke(s: str) -> bool:
    s = normalize_one_line(s)
    low = s.lower()

    if not (60 <= len(s) <= 220):
        return False

    if any(t in low for t in ["<im_start>", "<im_end>", "assistant", "gpt", "system", "user"]):
        return False
    if "http://" in low or "https://" in low:
        return False

    if s.startswith(("- ", "— ", "-\t", "—\t")):
        return False

    dialogs = s.count("—") + s.count("- ")
    sentences = sum(s.count(x) for x in [".", "!", "?"])
    if not (dialogs >= 2 or sentences >= 2):
        return False

    if not any(p in s for p in ["—", "?", "!"]):
        return False
    
    if is_nsfw(s):
        return False


    bad_starts = (
        "вот уже", "если у нас", "новый год", "жизнь", "социальные сети",
        "мне кажется", "я думаю", "важно", "нужно", "удобно", "сегодня"
    )
    if low.startswith(bad_starts):
        return False

    return True

def ok_row(x):
    joke = extract_gpt_joke(x)
    score = x.get("score", None)
    if score is not None:
        try:
            if float(score) < 4.7:  
                return False
        except:
            pass
    return is_ok_joke(joke)

def clean_joke_text(s: str) -> str:
    s = normalize_one_line(s)

    garbage_patterns = [
        r"анекдоты\s+и\s+шутки.*$",
        r"standup.*$",
        r"чат\s*•\s*истории.*$",
        r"^анекдоты\s*[:\-]\s*",      
    ]
    low = s.lower()
    for pat in garbage_patterns:
        s = re.sub(pat, "", s, flags=re.IGNORECASE).strip()

    s = re.sub(r"\bчат\b.*$", "", s, flags=re.IGNORECASE).strip()

    return s



In [ ]:
ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
print("Loaded:", DATASET_ID, "len=", len(ds))
print("Columns:", ds.column_names)
print("Example:", ds[0])

filtered = ds.filter(ok_row)  
filtered = filtered.shuffle(seed=42)

print("Filtered len:", len(filtered))
for i in range(3):
    print("SAMPLE", i, "=>", extract_gpt_joke(filtered[i]))

total = len(filtered)
train_n = min(TRAIN_SAMPLES, max(2000, total - EVAL_SAMPLES))
eval_n  = min(EVAL_SAMPLES, max(200, total - train_n))

train_raw = filtered.select(range(train_n))
eval_raw  = filtered.select(range(train_n, train_n + eval_n))

tokenizer_tmp = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer_tmp.pad_token is None:
    tokenizer_tmp.pad_token = tokenizer_tmp.eos_token

def make_chat_text(joke: str, tokenizer) -> dict:
    joke = normalize_one_line(joke)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
        {"role": "assistant", "content": joke},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = train_raw.map(lambda x: make_chat_text(extract_gpt_joke(x), tokenizer_tmp),
                         remove_columns=train_raw.column_names)
eval_ds  = eval_raw.map(lambda x: make_chat_text(extract_gpt_joke(x), tokenizer_tmp),
                        remove_columns=eval_raw.column_names)

print("train:", len(train_ds), "eval:", len(eval_ds))
print("\nTRAIN TEXT SAMPLE:\n", train_ds[0]["text"])

Loaded: samedad/mem-and-russian-jokes-dataset len= 521904
Columns: ['conversations', 'source', 'score', '__index_level_0__']
Example: {'conversations': [{'from': 'human', 'value': 'Хочу услышать шутку'}, {'from': 'gpt', 'value': 'Стиптизер по кличке Сусанин заводит только поляков'}], 'source': 'russian_jokes', 'score': 5.0, '__index_level_0__': 0}
Filtered len: 70551
SAMPLE 0 => Встречаются две подруги и одна другой говорит: "У нас вчера с мужем такой конфуз вышел! Представляешь, сидим мы, и тут - стук в дверь. А мы оба по привычке в шкаф как ломанемся!".
SAMPLE 1 => Владелец новой машины проснулся ночью от испуганных воплей сигнализации.Выбегает на улицу, а у подъезда два амбала стоят:- Что, ключи принес, братан? Ну, давай!
SAMPLE 2 => Две подруги:- Ты выходишь за него из-за денег?- Вовсе нет! Я даже не знаю точно, сколько у него миллионов.
train: 30000 eval: 1500

TRAIN TEXT SAMPLE:
 <|im_start|>system
Ты — профессиональный автор анекдотов. Пиши коротко, связно и смешно.<|im_end|>
<|

In [ ]:
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32),
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
else:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    totalp = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {totalp:,} ({100*trainable/totalp:.6f}%)")

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
model.config.use_cache = False

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
# ===== ЭКОНОМИЯ VRAM =====
model.gradient_checkpointing_enable()
model.config.use_cache = False
# ========================

TEXT_FIELD = "text"
cfg_sig = inspect.signature(SFTConfig.__init__).parameters

cfg_kwargs = dict(
    output_dir=OUTPUT_DIR,
    packing=PACKING,
    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,

    warmup_ratio=0.03,
    logging_steps=10,
    log_level="info",
    disable_tqdm=False,

    eval_steps=1000,
    save_steps=500,
    save_total_limit=2,

    eval_strategy="steps",
    report_to="none",
    remove_unused_columns=False,

    max_steps=MAX_STEPS,
    learning_rate=LR,

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=False,
)

if "max_grad_norm" in cfg_sig:
    cfg_kwargs["max_grad_norm"] = 1.0

if "max_length" in cfg_sig:
    cfg_kwargs["max_length"] = MAX_SEQ_LEN
elif "max_seq_length" in cfg_sig:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LEN

if "dataset_text_field" in cfg_sig:
    cfg_kwargs["dataset_text_field"] = TEXT_FIELD

cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if k in cfg_sig}
sft_args = SFTConfig(**cfg_kwargs)

trainer_kwargs = dict(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
if "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer

if "dataset_text_field" in trainer_sig:
    trainer_kwargs["dataset_text_field"] = TEXT_FIELD
elif "formatting_func" in trainer_sig:
    trainer_kwargs["formatting_func"] = lambda x: x[TEXT_FIELD]

trainer = SFTTrainer(**trainer_kwargs)

train_result = trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Train done. Output dir:", OUTPUT_DIR)
print(train_result)


Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-f

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1000,1.098400,1.065450,1.088684,5466191.000000,0.781480


Saving model checkpoint to ./joke-sft-qwen\checkpoint-500
loading configuration file config.json from cache at C:\Users\Евгений\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct\snapshots\7ae557604adf67be50417f59c2c2f167def9a775\config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

Train done. Output dir: ./joke-sft-qwen
TrainOutput(global_step=1500, training_loss=1.1192997983296713, metrics={'train_runtime': 10567.2947, 'train_samples_per_second': 4.542, 'train_steps_per_second': 0.142, 'total_flos': 1.803577588443264e+16, 'train_loss': 1.1192997983296713, 'epoch': 1.863421073064679})


In [ ]:
# после рестарта kernel
import os, glob, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, logging
from peft import PeftModel

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./joke-sft-qwen"

ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
               key=lambda p: int(os.path.basename(p).split("-")[-1]))
ADAPTER_DIR = ckpts[-1] if ckpts else OUTPUT_DIR
print("Adapter:", ADAPTER_DIR)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16,
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()
model.config.use_cache = True

SYSTEM_PROMPT = "Ты — профессиональный автор анекдотов. Пиши коротко, связно и смешно."
USER_PROMPT = "Напиши ОДИН анекдот на русском в 1–2 предложениях: сначала сетап, потом панчлайн. Только анекдот, без объяснений. Одна строка."

@torch.no_grad()
def gen_one(seed):
    set_seed(seed)
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":USER_PROMPT}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=True,
        temperature=0.95,
        top_p=0.92,
        repetition_penalty=1.18,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return " ".join(tokenizer.decode(gen_ids, skip_special_tokens=True).strip().split())

for s in [123,124,125]:
    print("\n---", s, "---")
    print(gen_one(s))


e:\Spbu_ML\spbu_dl_2025\venv_gpt2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Adapter: ./joke-sft-qwen\checkpoint-1500

--- 123 ---
В магазине:- Скажите, у вас есть конфеты с золотой флейт?- Да!- А что это за цвет?

--- 124 ---
Врач:- Слушай, ты не знаешь, куда я ухожу?- Не знаю!- Тогда докажи мне, где мы?

--- 125 ---
Самолет улетает в Лондон, приземляется во Франции.- Кто это?- Моя жена.- Что-что, а я же сама - жена...- Ну хорошо. Давай с тобой снимаем кабинки!


In [ ]:
import re, random, torch
from transformers import set_seed

PREFIXES_PATH = "prefixes.txt"
N_EXAMPLES = 10
BATCH = 8
MAX_TRIES = 8

MAX_NEW_TOKENS = 72
TEMPERATURE = 0.40
TOP_P = 0.90
TOP_K = 35
REP_PENALTY = 1.25
NO_REPEAT_NGRAM = 4

SEED = 42
set_seed(SEED)
random.seed(SEED)

SYSTEM_PROMPT = (
    "Ты — профессиональный автор русских анекдотов. "
    "Пиши только по-русски, связно и смешно."
)

USER_PROMPT = (
    "Продолжи и закончи ОДИН анекдот.\n"
    "Требования:\n"
    "1) одна строка\n"
    "2) 1–3 предложения\n"
    "3) в конце панчлайн\n"
    "4) запрещено использовать слова: Human, Assistant, System, User, GPT\n"
    "5) без объяснений и списков\n"
)

BAD_SUBSTR = [
    "system", "user", "assistant", "human", "gpt",
    "http://", "https://", "<|im_start|>", "<|im_end|>"
]

BAD_TECH = ("вопрос:", "ответ:", "характеристика:", "аппаратура:", "архитектур", "конфигурац", "структура")
BAD_STARTS = ("вот уже", "если у нас", "я думаю", "мне кажется", "важно", "нужно", "удобно")

def load_prefixes(path: str):
    prefs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            line = re.sub(r"^\d+\s*", "", line).strip()
            if line:
                prefs.append(line)
    return prefs

def one_line(s: str) -> str:
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def post_clean(s: str) -> str:
    s = re.sub(r"\b(Human|Assistant|System|User)\s*:\s*", "", s, flags=re.IGNORECASE)
    s = one_line(s)
    return s

def russianish_ok(s: str) -> bool:
    allowed = set(".,!?—-:;\"'()«» ")
    bad = 0
    for ch in s:
        if ch.isalnum() or ch in allowed or ch.isspace():
            continue
        bad += 1
    return bad / max(1, len(s)) <= 0.02

def looks_ok(joke: str, prefix: str) -> bool:
    j = one_line(joke)
    low = j.lower()

    if not j.startswith(prefix):
        return False
    if any(x in low for x in BAD_SUBSTR):
        return False
    if low.startswith(BAD_STARTS):
        return False
    if any(x in low for x in BAD_TECH):
        return False
    if not (45 <= len(j) <= 240):
        return False
    if not any(p in j for p in ".!?"):
        return False
    if j[-1] not in ".!?":
        return False
    if not russianish_ok(j):
        return False
    return True

base_msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]
prompt_prefix = tokenizer.apply_chat_template(base_msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_batch(prefix_list, seed):
    set_seed(seed)
    prompts = [prompt_prefix + p for p in prefix_list]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=REP_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    tails = []
    for i in range(out.shape[0]):
        gen_ids = out[i][inputs["input_ids"][i].shape[0]:]
        tails.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return tails

prefixes = load_prefixes(PREFIXES_PATH)
print("Loaded prefixes:", len(prefixes))
print("Example prefixes:", prefixes[:5])

result = []
seed_step = 0

while len(result) < N_EXAMPLES:
    need = N_EXAMPLES - len(result)
    cur_bs = min(BATCH, need)

    pref_batch = [random.choice(prefixes) for _ in range(cur_bs)]
    ok_mask = [False] * cur_bs
    best = [""] * cur_bs

    for _ in range(MAX_TRIES):
        tails = generate_batch(pref_batch, seed=SEED + seed_step)
        seed_step += 1

        for i, (pref, tail) in enumerate(zip(pref_batch, tails)):
            if ok_mask[i]:
                continue
            joke = post_clean(pref + " " + tail)
            if looks_ok(joke, pref):
                ok_mask[i] = True
                best[i] = joke

        if all(ok_mask):
            break

    for j in best:
        if j:
            result.append(j)
        if len(result) >= N_EXAMPLES:
            break

for i, j in enumerate(result, 1):
    print(f"\n{i}) {j}")







Loaded prefixes: 75
Example prefixes: ['Идёт мужик по лесу', 'Встречаются два друга', 'Приходят мужик в бар', 'Жена говорит мужу', 'Приходят альфа, бета и гамма в бар']

1) Жена спрашивает у мужа - Как дела? Мужик: - Хорошая! Женщина: - Спасибо за это!

2) Жена говорит мужу Вовочке:- Дорогой, а ты знаешь, что я сейчас не могу прийти домой?- Не знаю.- А почему же?- Да потому что мне нужно было выразить свою любовь к Вовочку!

3) Пишет код программист Вовочка на компьютере:- Давайте создадим сценарий игры!Сначала мы должны сделать две строчки.- А что делать дальше?

4) Встречаются два математика Приходит один к другому:- Товарищ, я не могу разгадать этот уравнение!Сообщается:- Да? Я же говорил вам, что это простое уравненство...

5) Идёт по лесу шаман специст. Мимо проходит крокодил:- Видишь, у меня на голове зеленая кожа!- А ты что, не видела? Это же черная!

6) Приходит мужик в аптеку Врача:- Доктор! Я ухожу на съезд.- Что случилось?- У меня болит голова...- А что вы делаете? - спросил

In [ ]:
import re, torch, random
from transformers import set_seed

PREFIXES_PATH = "prefixes.txt"
OUT_PATH = "joke.txt"

JOKES_PER_PREFIX = 2    
MAX_TRIES = 12           
BATCH = 6              

MAX_NEW_TOKENS = 96
MIN_NEW_TOKENS = 30

TEMPERATURE = 0.40
TOP_P = 0.90
TOP_K = 35
REP_PENALTY = 1.25
NO_REPEAT_NGRAM = 4

SEED = 42
set_seed(SEED)
random.seed(SEED)

SYSTEM_PROMPT = (
    "Ты — профессиональный автор русских анекдотов. "
    "Пиши только по-русски. Шутка должна быть связной и законченной."
)

USER_PROMPT = (
    "Продолжи и закончи ОДИН анекдот.\n"
    "Правила:\n"
    "- 1 строка\n"
    "- 1–3 предложения\n"
    "- в конце панчлайн\n"
    "- не используй слова: Human, Assistant, System, User, GPT\n"
    "- без объяснений, без списков\n"
)

BAD_SUBSTR = ["system", "user", "assistant", "human", "gpt", "http://", "https://", "<|im_start|>", "<|im_end|>"]

def one_line(s: str) -> str:
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def post_clean(s: str) -> str:
    s = re.sub(r"\b(Human|Assistant|System|User)\s*:\s*", "", s, flags=re.IGNORECASE)
    return one_line(s)

def parse_prefixes(path: str):
    res = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(r"^\s*(\d+)\s+(.*\S)\s*$", line)
            if not m:
                continue
            pid = int(m.group(1))
            ptxt = m.group(2).strip()
            res.append((pid, ptxt))
    return res

def strip_prefix(full: str, prefix_text: str) -> str:
    s = one_line(full)
    if s.startswith(prefix_text):
        s = s[len(prefix_text):].lstrip()
    s = re.sub(r"^(—|-|:)\s*", "", s).strip()
    return s

def looks_ok(full: str, prefix_text: str) -> bool:
    s = one_line(full)
    low = s.lower()

    if not s.startswith(prefix_text):
        return False
    if any(b in low for b in BAD_SUBSTR):
        return False
    if not (60 <= len(s) <= 320):  
        return False
    if not any(p in s for p in ".!?"):
        return False
    if s[-1] not in ".!?":
        return False
    lat = sum(ch.isascii() and ch.isalpha() for ch in s)
    if lat / max(1, len(s)) > 0.05:
        return False
    return True

base_msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]
prompt_prefix = tokenizer.apply_chat_template(base_msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_tails(prefix_texts, seed):
    set_seed(seed)
    prompts = [prompt_prefix + p for p in prefix_texts]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens=MIN_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=REP_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    tails = []
    for i in range(out.shape[0]):
        gen_ids = out[i][inputs["input_ids"][i].shape[0]:]
        tails.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return tails

prefix_pairs = parse_prefixes(PREFIXES_PATH)
print("Loaded prefixes:", len(prefix_pairs))
print("First 5:", prefix_pairs[:5])

lines = []
seed_step = 0

for pid, ptxt in prefix_pairs:
    got = 0
    best_fallback = None

    tries = 0
    while got < JOKES_PER_PREFIX and tries < MAX_TRIES:
        tries += 1
        cur_bs = min(BATCH, JOKES_PER_PREFIX - got)
        tails = generate_tails([ptxt]*cur_bs, seed=SEED + seed_step)
        seed_step += 1

        for tail in tails:
            full = post_clean(ptxt + " " + tail)
            cont = strip_prefix(full, ptxt)

            if cont and best_fallback is None:
                best_fallback = cont

            if looks_ok(full, ptxt) and cont:
                lines.append(f"{pid} {cont}")
                got += 1

            if got >= JOKES_PER_PREFIX:
                break

    if got == 0:
        lines.append(f"{pid} {best_fallback if best_fallback else '...'}")

present = set(int(re.match(r"^(\d+)\s", x).group(1)) for x in lines if re.match(r"^\d+\s", x))
wanted = [pid for pid, _ in prefix_pairs]
missing = [pid for pid in wanted if pid not in present]
print("Total lines:", len(lines))
print("Unique prefix ids in output:", len(present), "/", len(wanted))
print("Missing ids:", missing)

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for line in lines:
        f.write(one_line(line) + "\n")

print("Saved:", OUT_PATH)
print("Sample:\n", "\n".join(lines[:10]))




Loaded prefixes: 75
First 5: [(1, 'Идёт мужик по лесу'), (2, 'Встречаются два друга'), (3, 'Приходят мужик в бар'), (4, 'Жена говорит мужу'), (5, 'Приходят альфа, бета и гамма в бар')]
Total lines: 148
Unique prefix ids in output: 75 / 75
Missing ids: []
Saved: joke.txt
Sample:
 1 , смотрит на кустарника:- А ты что, арбуз?- Нет.- И как же я это понимал?
1 с собакой.- Дорогая! А ты где нашёла? - Да вот, у нас там кустарник. Смотри, он на дереве стоит...
2 .- Ты чего? - спрашивает он.- А ты что, смотри на меня?- Да нет... Смотри как я работаю!
2 .- А ты знаешь, какая твоя жена?- Да нет... Я так понимал, что она кричит "Красная"!
3 с девушкой.- Девушку у меня нет! У вас есть?- Да, но она там?..- Нет...- А что же ты ее так ждал?!
3 и говорит:- Скажите, у вас есть шампанское?- Нет.- А вы знаете почему?- Мне кажется, что это кашляло...
4 - Я хочу сексом.- А ты хочешь?- Нет... Я хотела бы! Но я не могу!- Тогда попробую еще раз - а ты хочешь?
4 - Сколько тебе лет?- Два.- А скольки раз ты меня 